# Notebook 40_balanced_supcon — Stage-A balanced-SupCon ablation

This is a strict representation-objective ablation of the current `31_frozen_encoder_expert` winner. The candidate keeps the same `47 → 128 → 64` shared encoder, four linear `64 → 22` experts, dropout-free `64 → 32 → 4` gate, optimizer settings, and frozen-body Stage C. The only training-contract change is Stage A: `sqrt-frequency-weighted CE` becomes `sqrt-frequency-weighted CE + 0.1 × balanced SupCon` at temperature `0.1`. Legacy shuffled sampling is intentionally retained so sampling is not a second experimental variable.

Because the contrastive objective changes the encoder coordinates, the candidate must retrain Stage A and its Stage-B experts. Stage C then freezes that newly trained encoder and expert bank and trains only the same 2,212 gate parameters. The existing winner is used as the paired reference and is never retrained here.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
REFERENCE_PREFIX = 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_v1'
REFERENCE_CONFIG = 'config/compact_soft_moe_deeper_gate_frozen_stagec_3seed.yaml'
CANDIDATE_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_deeper_gate32_frozen_stagec_v1'
CANDIDATE_CONFIG = 'config/compact_soft_moe_balanced_supcon_frozen_deeper_gate_3seed.yaml'
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS = 2  # Matches the currently used paired evidence; raise to 3 for a later confirmation.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight and candidate execution

The preflight proves that the deployed topology and Stages B/C are unchanged, while Stage A changes only `objective: ce` to `objective: balanced_supcon`. The candidate intentionally has no Stage-A/B reuse prefixes and therefore schedules all three stages.

In [ ]:
import copy, torch, yaml
from models.encoder import build_encoder
from models.representation_losses import StageARepresentationObjective
from training.logging_utils import run_streaming_logged
from training.model_utils import apply_stage_c_trainability, build_model
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
reference = yaml.safe_load(Path(REFERENCE_CONFIG).read_text())
candidate = yaml.safe_load(Path(CANDIDATE_CONFIG).read_text())
assert reference['seeds'] == candidate['seeds'] == [0, 1, 2]
assert 'reuse_stage_a_from_prefix' not in candidate and 'reuse_stage_b_from_prefix' not in candidate
ref = copy.deepcopy(reference['backbone']); cand = copy.deepcopy(candidate['backbone'])
ref_representation = ref['training']['representation']; cand_representation = cand['training']['representation']
assert ref_representation['objective'] == 'ce' and cand_representation['objective'] == 'balanced_supcon'
assert cand_representation['weight'] == 0.1 and cand_representation['temperature'] == 0.1
assert cand_representation['sampling'] == 'legacy' and cand_representation['class_weighting'] == 'legacy'
ref_representation['objective'] = cand_representation['objective']
assert ref == cand, 'Stage-A objective must be the only backbone-contract change'
model_cfg = candidate['backbone']['model']; training_cfg = candidate['backbone']['training']
encoder = build_encoder(47, 64, model_cfg['encoder'])
model = build_model(candidate['backbone']['architecture'], encoder, ['UNSW', 'ToN', 'BoT', 'CIC'], [f'class_{i}' for i in range(22)], model_cfg)
apply_stage_c_trainability(model, candidate['backbone'])
assert sum(p.numel() for p in model.parameters()) == candidate['expected_total_parameters'] == 22332
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 2212
objective = StageARepresentationObjective(64, 22, candidate['backbone'])
assert objective.objective == 'balanced_supcon'
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_representation_losses.py', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', CANDIDATE_CONFIG, '--prefix', CANDIDATE_PREFIX]
run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label='40-bsupcon-dry-run')
if EXECUTE: run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label='40-bsupcon-training')
else: print('NO TRAINING WAS STARTED. Review the contract, set EXECUTE=True, and rerun this cell.')

## Paired validation-first comparison

The analysis works after any common completed seeds rather than requiring a three-seed `Final_Study_Manifest.json`. Validation macro-F1 remains the declared primary endpoint. Accuracy and the dataset/rare-class guardrails help distinguish broad improvement from a narrow macro-F1 movement; locked-test metrics are descriptive only.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, display
conditions = {'reference_ce': REFERENCE_PREFIX, 'balanced_supcon': CANDIDATE_PREFIX}
summary_manifests = {name: (Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' / 'Final_Study_Manifest.json').is_file() for name, prefix in conditions.items()}
print('Three-seed summary completion:', summary_manifests)
validation_rows = []; test_rows = []; class_rows = []; dataset_rows = []; resource_rows = []
for condition, prefix in conditions.items():
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        required = ['manifest.json', 'Validation_Overall_Metrics.csv', 'Validation_Per_Dataset_Metrics.csv', 'Validation_Per_Class_Metrics.csv', 'Overall_Metrics.csv', 'Per_Class_Metrics.csv', 'Per_Dataset_Metrics.csv', 'Resource_Accounting.csv']
        if not all((result_dir / name).is_file() for name in required): continue
        overall_val = pd.read_csv(result_dir / 'Validation_Overall_Metrics.csv').iloc[0]
        val_ds = pd.read_csv(result_dir / 'Validation_Per_Dataset_Metrics.csv')
        val_cls = pd.read_csv(result_dir / 'Validation_Per_Class_Metrics.csv')
        rare = val_cls[(val_cls['support'] >= 20) & (val_cls['support'] < 200)]
        validation_rows.append({'condition': condition, 'seed': seed, 'validation_macro_f1': overall_val['macro_f1'], 'validation_accuracy': overall_val['accuracy'], 'worst_dataset_macro_f1': val_ds['macro_f1'].min(), 'rare_recall_mean': rare['recall'].mean() if len(rare) else np.nan, 'weakest_class_recall': val_cls.loc[val_cls['support'] > 0, 'recall'].min()})
        overall_test = pd.read_csv(result_dir / 'Overall_Metrics.csv').query("origin == 'ALL'").copy(); overall_test.insert(0, 'seed', seed); overall_test.insert(0, 'condition', condition); test_rows.append(overall_test)
        per_class = pd.read_csv(result_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL'").copy(); per_class.insert(0, 'seed', seed); per_class.insert(0, 'condition', condition); class_rows.append(per_class)
        per_dataset = pd.read_csv(result_dir / 'Per_Dataset_Metrics.csv').copy(); per_dataset.insert(0, 'seed', seed); per_dataset.insert(0, 'condition', condition); dataset_rows.append(per_dataset)
        resources = pd.read_csv(result_dir / 'Resource_Accounting.csv').copy(); resources['seed'] = seed; resources['condition'] = condition; resource_rows.append(resources)
validation = pd.DataFrame(validation_rows)
if validation.empty: print('No completed reference/candidate result pair exists yet.')
else:
    common = sorted(set(validation.query("condition == 'reference_ce'")['seed']) & set(validation.query("condition == 'balanced_supcon'")['seed']))
    print('Completed paired seeds:', common)
    paired_validation = validation[validation['seed'].isin(common)].copy(); display(paired_validation)
    for metric in ['validation_macro_f1', 'validation_accuracy', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class_recall']:
        pivot = paired_validation.pivot(index='seed', columns='condition', values=metric)
        pivot['delta_bsupcon_minus_ce'] = pivot['balanced_supcon'] - pivot['reference_ce']
        print(metric); display(pivot); print('Mean paired delta:', pivot['delta_bsupcon_minus_ce'].mean())
    if common and test_rows:
        test = pd.concat(test_rows, ignore_index=True); test = test[test['seed'].isin(common)]
        metrics = [c for c in ['accuracy', 'micro_f1', 'weighted_f1', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
        print('=== Locked-test descriptive metrics; never used for selection ==='); display(test.groupby('condition')[metrics].agg(['mean', 'std']))
        print('=== Per-dataset test macro-F1 ==='); display(pd.concat(dataset_rows, ignore_index=True).query('seed in @common').pivot_table(index='dataset', columns='condition', values='macro_f1', aggfunc='mean'))
        print('=== Per-class test F1 ==='); display(pd.concat(class_rows, ignore_index=True).query('seed in @common').pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    if resource_rows:
        resources = pd.concat(resource_rows, ignore_index=True); display(resources)
        candidate_resources = resources.query("condition == 'balanced_supcon'")
        assert set(candidate_resources['total_parameters'].astype(int)) == {22332}
        assert set(candidate_resources['trainable_parameters'].astype(int)) == {2212}

## Contrastive coverage, expert transfer, and freeze integrity

Coverage reveals how often Stage-A anchors actually had same-class positives under unchanged legacy sampling. Stage-B tables show whether the new latent space helped the linear experts before routing. Finally, every candidate Stage-C owned-expert metric must exactly equal its own Stage-B value, proving that Stage C remained gate-only.

In [ ]:
baseline_stage_b_prefix = 'nfv3_4way_moe_soft_comparable_repro_v1'
for seed in (0, 1, 2):
    result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_seed{seed}'
    if not (result_dir / 'manifest.json').is_file(): continue
    coverage_path = result_dir / 'Representation_Batch_Coverage.csv'
    if coverage_path.is_file():
        print(f'=== Seed {seed}: Stage-A contrastive coverage ==='); display(pd.read_csv(coverage_path))
    reference_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{baseline_stage_b_prefix}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
    candidate_b_path = Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{CANDIDATE_PREFIX}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv'
    candidate_b = pd.read_csv(candidate_b_path)
    stage_b_comparison = reference_b.merge(candidate_b, on='dataset', suffixes=('_ce', '_bsupcon'))
    for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'): stage_b_comparison[f'delta_{metric}'] = stage_b_comparison[f'{metric}_bsupcon'] - stage_b_comparison[f'{metric}_ce']
    print(f'=== Seed {seed}: owned-expert Stage-B transfer ==='); display(stage_b_comparison)
    candidate_c = pd.read_csv(result_dir / 'Validation_Expert_Owned_StageC.csv')
    preserved = candidate_b.merge(candidate_c, on='dataset', suffixes=('_stage_b', '_stage_c'))
    delta_columns = []
    for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'):
        column = f'delta_{metric}'; preserved[column] = preserved[f'{metric}_stage_c'] - preserved[f'{metric}_stage_b']; delta_columns.append(column)
    assert np.allclose(preserved[delta_columns], 0.0, atol=1e-12), f'Frozen-body drift in seed {seed}'
    print(f'=== Seed {seed}: exact candidate Stage-B competence preservation ==='); display(preserved)
    print(f'=== Seed {seed}: gate by source dataset ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
    print(f'=== Seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))

## Training and latent-space diagnostics

In [ ]:
candidate_summary = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_summary'
for seed in (0, 1, 2):
    result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_seed{seed}'
    history_path = result_dir / 'training' / 'Training_History.csv'
    if not history_path.is_file(): continue
    history = pd.read_csv(history_path)
    print(f'=== Seed {seed}: Stage A ==='); display(history.query("stage == 'A'")[[c for c in ['epoch', 'objective', 'train_ce_loss', 'train_representation_loss', 'train_total_loss'] if c in history]])
    print(f'=== Seed {seed}: Stage C ==='); display(history.query("stage == 'C'")[[c for c in ['epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left'] if c in history]])
if (candidate_summary / 'Final_Study_Manifest.json').is_file():
    display(pd.read_csv(candidate_summary / 'Latent_Snapshot_Summary_3Seed.csv'))
    for figure in sorted((candidate_summary / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))

## Interpretation rule

Treat balanced SupCon as helpful only when its paired validation effect is repeatable and the Stage-B expert, worst-dataset, and rare-class diagnostics do not reveal a compensating regression. If contrastive coverage is poor, this run tests the loss under legacy sampling but does not settle whether a class/domain-balanced sampler would help; that would require a separate sampling-matched CE control.